# Intelligent Loan Approval - EDA & Model Training
This notebook handles the data preprocessing, model training (Random Forest), evaluation, and exporting of the model artifacts to your Google Drive.

In [ ]:
!pip install xgboost imbalanced-learn shap -q

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import shap
from google.colab import drive

## 1. Mount Google Drive

In [ ]:
drive.mount('/content/drive')
# Create the destination directory if it doesn't exist
drive_path = '/content/drive/MyDrive/Intelligent Loan Approval'
os.makedirs(drive_path, exist_ok=True)

## 2. Load Data
**IMPORTANT**: Please ensure you have uploaded the `loan_dataset.csv` (Credit Risk Dataset from Kaggle) into your Colab environment or Google Drive and update the path below if necessary.

In [ ]:
# Assuming the dataset is uploaded to the root of the Colab session
df = pd.read_csv('loan_dataset.csv')
df.head()

## 3. Data Preprocessing

In [ ]:
# Drop nulls (or impute them)
df = df.dropna()
df = df.drop_duplicates()

# Separate features and target (Assuming target is 'loan_status')
X = df.drop('loan_status', axis=1)
y = df['loan_status']

numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# 1. Scale numerical features
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X[numerical_cols])

# 2. Encode categorical features
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat_encoded = encoder.fit_transform(X[categorical_cols])
cat_feature_names = encoder.get_feature_names_out(categorical_cols)

# Combine features
X_processed = np.hstack((X_num_scaled, X_cat_encoded))
feature_columns = numerical_cols + list(cat_feature_names)

print(f'Processed shape: {X_processed.shape}')

## 4. Train/Test Split and Handling Class Imbalance

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42, stratify=y)

print("Before SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print(y_train_resampled.value_counts())

## 5. Model Training (Random Forest)

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=15)
model.fit(X_train_resampled, y_train_resampled)

## 6. Evaluation

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## 7. Model Export to Google Drive (with Accuracy Check)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")

if accuracy >= 0.90:
    user_choice = input(f"Model accuracy is {accuracy:.2%} (>= 90%). Do you want to export the models to Google Drive? (yes/no): ")
    if user_choice.strip().lower() in ['yes', 'y']:
        joblib.dump(model, f"{drive_path}/loan_model.pkl")
        joblib.dump(scaler, f"{drive_path}/scaler.pkl")
        joblib.dump(encoder, f"{drive_path}/encoder.pkl")
        joblib.dump(feature_columns, f"{drive_path}/feature_columns.pkl")
        print(f"Models successfully exported to: {drive_path}")
    else:
        print("Export cancelled by user.")
else:
    print(f"Model accuracy is {accuracy:.2%} which is below the 90% threshold. Models will not be exported.")